In [1]:
from google.colab import files
uploaded = files.upload()

Saving movie_keywords_cleaned.csv to movie_keywords_cleaned.csv
Saving movies_cleaned.csv to movies_cleaned.csv
Saving cast_cleaned.csv to cast_cleaned.csv
Saving crew_cleaned.csv to crew_cleaned.csv
Saving genres_cleaned.csv to genres_cleaned.csv
Saving movie_genres_cleaned.csv to movie_genres_cleaned.csv


In [6]:
!pip install streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 7.7 MB/s  0:00:00


In [2]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-11-14 16:34:55--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64 [following]
--2026-11-14 16:34:55--  https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/551ef38d-56cb-4deb-8ae1-554a6a8551e7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-14T17%3A31%3A09Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-14

In [3]:
%%writefile app.py

Writing app.py


In [7]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from streamlit_option_menu import option_menu


# Database connection
engine = create_engine("sqlite:///movies.db")

#reading the cleaned file
movies = pd.read_csv('movies_cleaned.csv')
genres = pd.read_csv('genres_cleaned.csv')
movie_genres = pd.read_csv('movie_genres_cleaned.csv')
cast = pd.read_csv('cast_cleaned.csv')
crew = pd.read_csv('crew_cleaned.csv')
movie_keywords = pd.read_csv('movie_keywords_cleaned.csv')

#Pushing dataset into database
movies.to_sql('movies', engine, if_exists='replace', index=False)
genres.to_sql('genres', engine, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', engine, if_exists='replace', index=False)
cast.to_sql('cast', engine, if_exists='replace', index=False)
crew.to_sql('crew', engine, if_exists='replace', index=False)
movie_keywords.to_sql('movie_keywords', engine, if_exists='replace', index=False)


st.title("TMDB Analysis")


# Convert release_date into datetime format
movies['release_date'] = pd.to_datetime(movies['release_date'], format = '%Y-%m-%d')
# Extract release year
movies["release_year"] = movies["release_date"].dt.year

# Find minimum and maximum release years
min_year = int(movies["release_year"].min())
max_year = int(movies["release_year"].max())

if "selected_genres" not in st.session_state:
    st.session_state.selected_genres = []

if "selected_years" not in st.session_state:
    st.session_state.selected_years = (min_year, max_year)

if "min_rating" not in st.session_state:
    st.session_state.min_rating = 0.0

if "selected_language" not in st.session_state:
    st.session_state.selected_language = "All Languages"

# Sidebar navigation
with st.sidebar:
    selected = option_menu(
        "Main Menu",
        ["Home", "Filters", "Dashboard", "Queries"],
        icons=["house", "sliders", "bar-chart", "search"],
        menu_icon="cast",
        default_index=0
    )

# Home page
if selected == "Home":
    st.write("Welcome to my project")


elif selected == "Filters":

    st.title("Movie Filters")

    # 1. Genre multiselect
    genre_data = pd.read_sql(
        """
        SELECT DISTINCT genre_name
        FROM genres
        ORDER BY genre_name;
        """,
        engine
    )

    st.multiselect(
        "Select Genre",
        options=genre_data["genre_name"].tolist(),
        key="selected_genres"
    )

    # 2. Release-year range
    st.slider(
        "Select Release Year Range",
        min_value=min_year,
        max_value=max_year,
        key="selected_years"
    )

    # 3. Minimum rating
    st.slider(
        "Select Minimum Rating",
        min_value=0.0,
        max_value=10.0,
        step=0.5,
        key="min_rating"
    )

    # 4. Language selectbox
    language_data = pd.read_sql(
        """
        SELECT DISTINCT original_language
        FROM movies
        WHERE original_language IS NOT NULL
        ORDER BY original_language;
        """,
        engine
    )

    language_list = ["All Languages"] + language_data["original_language"].tolist()

    if st.session_state.selected_language not in language_list:
        st.session_state.selected_language = "All Languages"

    st.selectbox(
        "Select Language",
        options=language_list,
        key="selected_language"
    )

    # 5. Live count of movies matching all selected filters
    filter_conditions = [
        "m.release_date IS NOT NULL",
        "CAST(strftime('%Y', m.release_date) AS INTEGER) BETWEEN ? AND ?",
        "m.vote_average >= ?"
    ]

    filter_params = [
        st.session_state.selected_years[0],
        st.session_state.selected_years[1],
        st.session_state.min_rating
    ]

    if st.session_state.selected_language != "All Languages":
        filter_conditions.append("m.original_language = ?")
        filter_params.append(st.session_state.selected_language)

    if st.session_state.selected_genres:
        genre_placeholders = ",".join(
            ["?"] * len(st.session_state.selected_genres)
        )
        filter_conditions.append(
            f"""EXISTS (
                SELECT 1
                FROM movie_genres AS mg
                JOIN genres AS g ON mg.genre_id = g.genre_id
                WHERE mg.movie_id = m.movie_id
                AND g.genre_name IN ({genre_placeholders})
            )"""
        )
        filter_params.extend(st.session_state.selected_genres)

    filters_where = " AND ".join(filter_conditions)

    count_query = f"""
        SELECT COUNT(*) AS total_movies
        FROM movies AS m
        WHERE {filters_where};
    """

    count_result = pd.read_sql(
        count_query,
        engine,
        params=filter_params
    )

    st.divider()
    st.metric(
        "Total Movies Matching Filters",
        int(count_result["total_movies"].iloc[0])
    )


elif selected == "Dashboard":

    st.title("TMDB Movie Dashboard")


    conditions = [
        "m.release_date IS NOT NULL",
        "CAST(strftime('%Y', m.release_date) AS INTEGER) BETWEEN ? AND ?",
        "m.vote_average >= ?"
    ]

    params = [
        st.session_state.selected_years[0],
        st.session_state.selected_years[1],
        st.session_state.min_rating
    ]

    if st.session_state.selected_language != "All Languages":
        conditions.append("m.original_language = ?")
        params.append(st.session_state.selected_language)

    if st.session_state.selected_genres:
        placeholders = ",".join(
            ["?"] * len(st.session_state.selected_genres)
        )
        conditions.append(
            f"""EXISTS (
                SELECT 1
                FROM movie_genres AS mg_filter
                JOIN genres AS g_filter
                    ON mg_filter.genre_id = g_filter.genre_id
                WHERE mg_filter.movie_id = m.movie_id
                AND g_filter.genre_name IN ({placeholders})
            )"""
        )
        params.extend(st.session_state.selected_genres)

    where_clause = " AND ".join(conditions)


    kpi_query = f"""
        SELECT
            COUNT(DISTINCT m.movie_id) AS total_movies,
            COUNT(DISTINCT mg.genre_id) AS total_genres,
            COUNT(DISTINCT c.person_id) AS total_actors,
            AVG(m.vote_average) AS average_rating
        FROM movies AS m
        LEFT JOIN movie_genres AS mg
            ON m.movie_id = mg.movie_id
        LEFT JOIN "cast" AS c
            ON m.movie_id = c.movie_id
        WHERE {where_clause};
    """

    kpi = pd.read_sql(kpi_query, engine, params=params)

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Movies", int(kpi["total_movies"].iloc[0]))
    col2.metric("Total Genres Represented", int(kpi["total_genres"].iloc[0]))
    col3.metric("Total Actors", int(kpi["total_actors"].iloc[0]))
    col4.metric(
        "Average Rating",
        round(float(kpi["average_rating"].iloc[0] or 0), 2)
    )

    st.divider()


    st.subheader("Movie Analytics")
    col1, col2, col3, col4 = st.columns(4)

    # Column 1: Top 10 movies by revenue
    with col1:
        st.markdown("**Top 10 Movies by Revenue**")

        revenue_query = f"""
            SELECT m.title, m.revenue
            FROM movies AS m
            WHERE {where_clause}
            ORDER BY m.revenue DESC
            LIMIT 10;
        """

        revenue_result = pd.read_sql(
            revenue_query,
            engine,
            params=params
        )

        if not revenue_result.empty:
            fig, ax = plt.subplots(figsize=(5, 5))
            sns.barplot(
                data=revenue_result,
                x="revenue",
                y="title",
                ax=ax
            )
            ax.set_xlabel("Revenue")
            ax.set_ylabel("Movie")
            ax.set_title("Top 10 by Revenue")
            ax.tick_params(axis="y", labelsize=7)
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("No movies match the selected filters.")

    # Column 2: Genre distribution
    with col2:
        st.markdown("**Genre Distribution**")

        genre_distribution_query = f"""
            SELECT
                g.genre_name,
                COUNT(DISTINCT m.movie_id) AS movie_count
            FROM movies AS m
            JOIN movie_genres AS mg
                ON m.movie_id = mg.movie_id
            JOIN genres AS g
                ON mg.genre_id = g.genre_id
            WHERE {where_clause}
            GROUP BY g.genre_name
            ORDER BY movie_count DESC;
        """

        genre_result = pd.read_sql(
            genre_distribution_query,
            engine,
            params=params
        )

        if not genre_result.empty:
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.pie(
                genre_result["movie_count"],
                labels=genre_result["genre_name"],
                autopct="%1.0f%%",
                startangle=90,
                textprops={"fontsize": 7}
            )
            ax.set_title("Genre Distribution")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("No genre data for this selection.")

    # Column 3: Budget versus revenue
    with col3:
        st.markdown("**Budget vs Revenue**")

        budget_revenue_query = f"""
            SELECT m.title, m.budget, m.revenue
            FROM movies AS m
            WHERE {where_clause}
            AND m.budget > 0
            AND m.revenue > 0;
        """

        budget_revenue_result = pd.read_sql(
            budget_revenue_query,
            engine,
            params=params
        )

        if not budget_revenue_result.empty:
            fig, ax = plt.subplots(figsize=(5, 5))
            sns.scatterplot(
                data=budget_revenue_result,
                x="budget",
                y="revenue",
                ax=ax
            )
            ax.set_xlabel("Budget")
            ax.set_ylabel("Revenue")
            ax.set_title("Budget vs Revenue")
            ax.tick_params(axis="x", labelrotation=45, labelsize=7)
            ax.tick_params(axis="y", labelsize=7)
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("No budget/revenue data for this selection.")

    # Column 4: Highest-grossing genre insight
    with col4:
        st.markdown("**Dashboard Insight**")

        highest_genre_query = f"""
            SELECT
                g.genre_name,
                SUM(m.revenue) AS total_revenue
            FROM movies AS m
            JOIN movie_genres AS mg
                ON m.movie_id = mg.movie_id
            JOIN genres AS g
                ON mg.genre_id = g.genre_id
            WHERE {where_clause}
            GROUP BY g.genre_name
            ORDER BY total_revenue DESC
            LIMIT 1;
        """

        highest_genre_result = pd.read_sql(
            highest_genre_query,
            engine,
            params=params
        )

        if (
            not highest_genre_result.empty
            and pd.notna(highest_genre_result["total_revenue"].iloc[0])
        ):
            highest_genre = highest_genre_result["genre_name"].iloc[0]
            total_revenue = highest_genre_result["total_revenue"].iloc[0]

            st.write(
                f"The highest-grossing genre in the selected movies is "
                f"**{highest_genre}**, with total revenue of "
                f"**{total_revenue:,.0f}**."
            )
        else:
            st.info("No revenue insight is available for this selection.")

elif selected == "Queries":

    st.title("Movie Query Explorer")


    selected_genres = st.session_state.selected_genres
    selected_years = st.session_state.selected_years
    min_rating = st.session_state.min_rating
    selected_language = st.session_state.selected_language

    option = st.selectbox(
        "Select the Query",
        (
            "List all the action movies released after 2015",
            "Find the top 10 highest-grossing movies along with genres",
            "Calculate the average budget and average revenue for each genre",
            "List the top 10 actors who have appeared in most movies",
            "Find all the director who have directed more than 3 moveies along with movie count",
            "List the top 10 most frequently used keyword across all movies",
            "Find movies where the budget was above overall average budget but revenue was below the overall average revenue",
            "List all actors who have appeared in a movie directed by a specific director",
            "Find the genre with highest average rating, only genre with atleast 20 movies",
            "List the top 10 movies with the highest number of keywords tagged to them"
        )
    )

    if option == "List all the action movies released after 2015":

        st.subheader("Action Movies Released After 2015")

        query = """
            SELECT m.title, m.release_date
            FROM movies AS m
            JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
            JOIN genres AS g ON mg.genre_id = g.genre_id
            WHERE g.genre_name = 'Action'
            AND m.release_date > '2015-12-31'
            ORDER BY m.release_date;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            result["release_date"] = pd.to_datetime(
                result["release_date"],
                errors="coerce"
            )
            result["release_year"] = result["release_date"].dt.year
            year_counts = result["release_year"].value_counts().sort_index()

            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                x=year_counts.index,
                y=year_counts.values,
                ax=ax
            )
            ax.set_xlabel("Release Year")
            ax.set_ylabel("Number of Movies")
            ax.set_title("Action Movies Released After 2015")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "Find the top 10 highest-grossing movies along with genres":

        st.subheader("Top 10 Highest-Grossing Movies")

        query = """
            SELECT
                m.title,
                m.revenue,
                GROUP_CONCAT(DISTINCT g.genre_name) AS genre_name
            FROM movies AS m
            JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
            JOIN genres AS g ON mg.genre_id = g.genre_id
            GROUP BY m.movie_id, m.title, m.revenue
            ORDER BY m.revenue DESC
            LIMIT 10;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(data=result, x="revenue", y="title", ax=ax)
            ax.set_xlabel("Movie Revenue")
            ax.set_ylabel("Movie Title")
            ax.set_title("Top 10 Highest-Grossing Movies")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "Calculate the average budget and average revenue for each genre":

        st.subheader("Average Budget and Revenue by Genre")

        query = """
            SELECT
                g.genre_name,
                AVG(m.budget) AS average_budget,
                AVG(m.revenue) AS average_revenue
            FROM movies AS m
            JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
            JOIN genres AS g ON mg.genre_id = g.genre_id
            GROUP BY g.genre_name
            ORDER BY average_revenue DESC;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(12, 6))
            result_melted = result.melt(
                id_vars="genre_name",
                value_vars=["average_budget", "average_revenue"],
                var_name="Type",
                value_name="Amount"
            )
            sns.barplot(
                data=result_melted,
                x="genre_name",
                y="Amount",
                hue="Type",
                ax=ax
            )
            ax.set_xlabel("Genre")
            ax.set_ylabel("Amount")
            ax.set_title("Average Budget and Average Revenue by Genre")
            ax.tick_params(axis="x", rotation=45)
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "List the top 10 actors who have appeared in most movies":

        st.subheader("Top 10 Actors by Movie Appearances")

        query = """
            SELECT
                actor_name,
                COUNT(DISTINCT movie_id) AS total_movies
            FROM "cast"
            GROUP BY person_id, actor_name
            ORDER BY total_movies DESC
            LIMIT 10;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                data=result,
                x="total_movies",
                y="actor_name",
                ax=ax
            )
            ax.set_xlabel("Number of Movies")
            ax.set_ylabel("Actor Name")
            ax.set_title("Top 10 Actors by Movie Appearances")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "Find all the director who have directed more than 3 moveies along with movie count":

        st.subheader("Directors with More Than 3 Movies")

        query = """
            SELECT
                person_name AS director_name,
                COUNT(DISTINCT movie_id) AS movie_count
            FROM crew
            WHERE job = 'Director'
            GROUP BY person_id, person_name
            HAVING COUNT(DISTINCT movie_id) > 3
            ORDER BY movie_count DESC;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                data=result,
                x="movie_count",
                y="director_name",
                ax=ax
            )
            ax.set_xlabel("Number of Movies Directed")
            ax.set_ylabel("Director Name")
            ax.set_title("Directors with More Than 3 Movies")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "List the top 10 most frequently used keyword across all movies":

        st.subheader("Top 10 Most Frequently Used Keywords")

        query = """
            SELECT
                keyword_name,
                COUNT(DISTINCT movie_id) AS movie_count
            FROM movie_keywords
            GROUP BY keyword_id, keyword_name
            ORDER BY movie_count DESC
            LIMIT 10;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.lineplot(
                data=result,
                x="keyword_name",
                y="movie_count",
                marker="o",
                ax=ax
            )
            ax.set_xlabel("Keyword")
            ax.set_ylabel("Number of Movies")
            ax.set_title("Top 10 Most Frequently Used Keywords")
            ax.tick_params(axis="x", rotation=45)
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "Find movies where the budget was above overall average budget but revenue was below the overall average revenue":

        st.subheader("Movies with High Budget and Low Revenue")

        query = """
            SELECT title, budget, revenue
            FROM movies
            WHERE budget > (SELECT AVG(budget) FROM movies)
            AND revenue < (SELECT AVG(revenue) FROM movies)
            ORDER BY budget DESC;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.scatterplot(
                data=result,
                x="budget",
                y="revenue",
                hue="title",
                ax=ax
            )
            ax.set_xlabel("Movie Budget")
            ax.set_ylabel("Movie Revenue")
            ax.set_title("Movies with Above-Average Budget and Below-Average Revenue")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "List all actors who have appeared in a movie directed by a specific director":

        st.subheader("Actors in Quentin Tarantino Movies")

        query = """
            SELECT
                c.actor_name,
                COUNT(DISTINCT c.movie_id) AS movie_count
            FROM "cast" AS c
            JOIN crew AS cr ON c.movie_id = cr.movie_id
            WHERE cr.job = 'Director'
            AND cr.person_name = 'Quentin Tarantino'
            GROUP BY c.person_id, c.actor_name
            ORDER BY movie_count DESC;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                data=result.head(10),
                x="movie_count",
                y="actor_name",
                ax=ax
            )
            ax.set_xlabel("Number of Movies")
            ax.set_ylabel("Actor Name")
            ax.set_title("Actors in Quentin Tarantino Movies")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "Find the genre with highest average rating, only genre with atleast 20 movies":

        st.subheader("Highest-Rated Genre")

        query = """
            SELECT
                g.genre_name,
                AVG(m.vote_average) AS average_rating
            FROM movies AS m
            JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
            JOIN genres AS g ON mg.genre_id = g.genre_id
            GROUP BY g.genre_name
            HAVING COUNT(DISTINCT m.movie_id) >= 20
            ORDER BY average_rating DESC
            LIMIT 1;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(8, 5))
            sns.barplot(
                data=result,
                x="genre_name",
                y="average_rating",
                ax=ax
            )
            ax.set_xlabel("Genre")
            ax.set_ylabel("Average Rating")
            ax.set_title("Genre with the Highest Average Rating")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)

    elif option == "List the top 10 movies with the highest number of keywords tagged to them":

        st.subheader("Top 10 Movies by Keyword Count")

        query = """
            SELECT
                m.title,
                COUNT(mk.keyword_id) AS keyword_count
            FROM movies AS m
            JOIN movie_keywords AS mk ON m.movie_id = mk.movie_id
            GROUP BY m.movie_id, m.title
            ORDER BY keyword_count DESC
            LIMIT 10;
        """

        result = pd.read_sql(query, engine)
        st.dataframe(result)

        if not result.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                data=result,
                x="keyword_count",
                y="title",
                ax=ax
            )
            ax.set_xlabel("Number of Keywords")
            ax.set_ylabel("Movie Title")
            ax.set_title("Top 10 Movies by Number of Keywords")
            plt.tight_layout()
            st.pyplot(fig)
            plt.close(fig)


2026-11-14 16:46:01.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 16:46:03.943 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 16:46:04.043 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-11-14 16:46:04.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 16:46:04.048 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 16:46:04.074 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 16:46:04.077 Session state does not function when running a script without `streamlit run`
2026-11-14 16:46:04.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14

In [8]:
!streamlit run /content/app.py &>/content/logs.txt &

In [9]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://infants-phi-mia-continuously.trycloudflare.com
